# Capacity expansion with InvestmentBlock

This notebook describes an example on how to use InvestmentBlock solver from [SMS++ tools](https://gitlab.com/smspp/tools) that allows using Benders Decomposition for capacity expansion, where investment decisions are located in the master problem and dispatch ones in the lower problem. To do so, the PyPSA model is converted to an [InvestmentBlock](https://gitlab.com/smspp/investmentblock) and then solved using the appropriate solver.

This notebook relies on two libraries:
- pySMSpp: it aims to provide an abstract python interface for SMS++ models
- pypsa2smspp: it aims to provide a converter from PyPSA to SMS++ models exploiting pySMSpp

In this notebook, the following steps are performed:
1. Create a simple PyPSA model
2. Define the configuration to use InvestmentBlock
3. Convert the PyPSA model to SMS++ using pypsa2smspp and optimize the model
4. Verify the equivalence of objective function
5. Analyze the PyPSA model created with SMS++ and pypsa2smspp and verify optimal investment decisions

## 1. Creation of the PyPSA model

We create a simple PyPSA model arbitrary number of buses, few generators, storages and loads.

Main assumptions are:
- The network is purely radial in the form bus1 -> bus2 -> bus3 -> ... busN
- A load is added to each bus of the network
- A generator is added to the first node

In [ ]:
import pypsa
import pandas as pd
import pypsa2smspp

#### The following code creates the desired pypsa model.

In [ ]:
n = pypsa.Network()
n.set_snapshots(pd.date_range("2024-01-01T00:00", "2024-01-01T23:00", freq="h"))

# Add carriers
n.add("Carrier", "AC")

# Add buses
n_buses = 2
for b in range(n_buses):
    n.add("Bus", f"bus{b}", carrier="AC")

# Add lines in a radial topology using bidirectional links
n_lines = n_buses - 1
for l in range(n_lines):
    n.add(
        "Link",
        f"line{l}",
        bus0=f"bus{l}",
        bus1=f"bus{l+1}",
        length=1,
        capital_cost=1000,
        p_min_pu=-1,
        p_nom_extendable=True,
    )

# Add a load to each bus
n_loads = n_buses
for l in range(n_loads):
    n.add("Load", f"load{l}", bus=f"bus{l}", p_set=pd.Series(100, index=n.snapshots))

# Add a generator to the first bus
n.add(
    "Generator",
    "gen0",
    bus="bus0",
    p_nom_extendable=True,
    capital_cost=1000,
    marginal_cost=1,
)

# Add a slack unit to each node: needed when using investmentblock
for b in range(n_buses):
    n.add(
        "Generator",
        f"slack{b}",
        bus=f"bus{b}",
        carrier="slack",
        marginal_cost=1000,
        p_nom=1e6,
    )

In [ ]:
n.optimize(solver_name="highs")

## 2. Define the configuration to use InvestmentBlock

In [ ]:
# Create the transformation object and run the transformation
# The `capacity_expansion_ucblock` option is set to `False` to enable capacity expansion using InvestmentBlockSolver instead of UnitCommitmentBlockSolver
tran = pypsa2smspp.Transformation(
    capacity_expansion_ucblock=False,
)

## 3. Convert the PyPSA model to SMS++ and optimize it

In [ ]:
n_smspp = tran.run(n, verbose=False)  # create the model and optimizes it in one line

# If you wish to create the model and optimize it in separate steps, you can do so as follows:
# tran.create_model(n)  # create the model
# tran.optimize(solver_name="highs")  # optimize the model
# n_smspp = tran.retrieve_solution(n)  # retrieve the solution and store it in a new Network object

Objective value

In [ ]:
n_smspp.objective

## 4. Verify the equivalence of SMS++ and PyPSA results

In [ ]:
pypsa_obj = n.objective
smspp_obj = n_smspp.objective
print("SMS++ obj         : %.6f" % smspp_obj)
print("PyPSA obj         : %.6f" % pypsa_obj)
print("Error SMS++ - PyPSA [%%]: %.5f" % (100*(smspp_obj - pypsa_obj)/pypsa_obj))

## 4. Analyze the PyPSA model created with SMS++ and pypsa2smspp

Retrieve PyPSA statistics

In [ ]:
n_stat = n_smspp.statistics.optimal_capacity()
n_stat

Retrieve SMS++ statistics

In [ ]:
n_parsed_stat = n_smspp.statistics.optimal_capacity()
n_parsed_stat

Calculate difference between the two

In [ ]:
error_stat = n_stat - n_parsed_stat

merged_stat = pd.concat([n_stat, n_parsed_stat, error_stat], axis=1)
merged_stat.columns = ["pypsa", "smspp", "difference"]
merged_stat